In [ ]:
import RPi.GPIO as GPIO
import time
import speech_recognition as sr
import pyttsx3
import cv2
import numpy as np
import tensorflow as tf
import json
import pickle
from picamera2 import Picamera2
import pygame

# === GPIO Setup ===
IN1 = 11
IN2 = 13
IN3 = 15
IN4 = 16
TRIG = 18
ECHO = 22

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BOARD)
GPIO.setup([IN1, IN2, IN3, IN4, TRIG], GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

# === TTS and Beep ===
engine = pyttsx3.init()
engine.setProperty('rate', 150)
pygame.mixer.init()
pygame.mixer.music.load("Downloads/beep-01a.wav")

# === Load Dataset and Model ===
with open("Downloads/file1.txt", "r") as f:
    robot_data = json.load(f)

model = tf.keras.models.load_model("robot_image_model.h5")
with open("label_classes.pkl", "rb") as f:
    label_classes = pickle.load(f)

# === Camera Initialization ===
picam2 = Picamera2()
picam2.start()
time.sleep(1)

# === Helper Functions ===
def beep():
    pygame.mixer.music.play()
    time.sleep(1)

def speak(text):
    print(text)
    engine.say(text)
    engine.runAndWait()

def listen():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("Listening...")
        recognizer.adjust_for_ambient_noise(source)
        try:
            audio = recognizer.listen(source, timeout=5)
            return recognizer.recognize_google(audio).lower()
        except:
            return ""

def move_forward():
    GPIO.output(IN1, GPIO.HIGH)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.HIGH)
    GPIO.output(IN4, GPIO.LOW)

def move_backward():
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.HIGH)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.HIGH)

def stop():
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.LOW)

def check_distance():
    GPIO.output(TRIG, False)
    time.sleep(0.05)

    GPIO.output(TRIG, True)
    time.sleep(0.00001)
    GPIO.output(TRIG, False)

    pulse_start, pulse_end = time.time(), time.time()
    while GPIO.input(ECHO) == 0:
        pulse_start = time.time()
    while GPIO.input(ECHO) == 1:
        pulse_end = time.time()

    pulse_duration = pulse_end - pulse_start
    distance = round(pulse_duration * 17150, 2)  # cm
    return distance

def describe_robot(name):
    for robot in robot_data:
        if name.lower() in robot["name"].lower():
            desc = f"{robot['name']} is manufactured by {robot['manufacturer']}. "
            if "weight_kg" in robot:
                desc += f"It weighs {robot['weight_kg']} kilograms. "
            if "payload_kg" in robot:
                desc += f"It can carry a payload of {robot['payload_kg']} kilograms. "
            if "degrees_of_freedom" in robot:
                desc += f"It has {robot['degrees_of_freedom']} degrees of freedom. "
            if "power_source" in robot:
                desc += f"It is powered by {robot['power_source']}. "
            if "applications" in robot:
                desc += f"It is used for {', '.join(robot['applications'])}. "
            if "control_methods" in robot:
                desc += f"It can be controlled using {', '.join(robot['control_methods'])}. "
            if "functions" in robot:
                desc += f"Its key functions include {', '.join(robot['functions'])}. "
            speak(desc)
            return
    speak("Sorry, I don't have information about this robot.")

def predict_robot():
    frame = picam2.capture_array()
    if frame is None or frame.size == 0:
        return None, 0.0

    image = cv2.resize(frame[:, :, :3], (224, 224)) / 255.0
    image = np.expand_dims(image, axis=0)
    predictions = model.predict(image)[0]
    idx = np.argmax(predictions)
    confidence = predictions[idx]
    return label_classes[idx], confidence

# === Main Function ===
def main():
    beep()
    speak("Hello! I am your Department Guide Robot. I can recognize Kinova, AR100, and Transmission Line robots. Please say the name of the robot or wait for me to detect one.")

    initial_command = listen()
    if any(robot.lower() in initial_command for robot in label_classes):
        describe_robot(initial_command)

    direction = "forward"

    while True:
        distance = check_distance()
        if distance < 20:
            speak("Too close! Reversing.")
            stop()
            move_backward()
            time.sleep(2)
            stop()
            continue

        speak("Detecting robot now.")
        name, confidence = predict_robot()

        if name and confidence > 0.4:
            speak(f"I think this is {name}. Let me tell you more.")
            describe_robot(name)
        else:
            speak("Sorry, I could not recognize the robot.")
            continue

        speak("Say stop, tell me about another robot, or say robot to continue.")
        user_cmd = listen()

        if "stop" in user_cmd:
            speak("Okay, stopping.")
            stop()
            break
        elif "another" in user_cmd or "about" in user_cmd:
            speak("Which robot?")
            robot_name = listen()
            describe_robot(robot_name)
        elif "robot" in user_cmd:
            speak("Okay, moving forward.")
            move_forward()
            time.sleep(2)
            stop()
        else:
            speak("I will now move backward.")
            move_backward()
            time.sleep(2)
            stop()

# === Run ===
try:
    main()
except KeyboardInterrupt:
    stop()
    GPIO.cleanup()
    picam2.stop()
    print("Program stopped.")


In [ ]:
import RPi.GPIO as GPIO
import time
import speech_recognition as sr
import pyttsx3
import cv2
import numpy as np
import tensorflow as tf
import json
import pickle
from picamera2 import Picamera2
import pygame

# === GPIO Setup ===
IN1 = 11
IN2 = 13
IN3 = 15
IN4 = 16
TRIG = 18
ECHO = 22

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BOARD)
GPIO.setup([IN1, IN2, IN3, IN4, TRIG], GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)

# === TTS and Beep ===
engine = pyttsx3.init()
engine.setProperty('rate', 150)
pygame.mixer.init()
pygame.mixer.music.load("Downloads/beep-01a.wav")

# === Load Dataset and Model ===
with open("Downloads/file1.txt", "r") as f:
    robot_data = json.load(f)

model = tf.keras.models.load_model("robot_image_model.h5")
with open("label_classes.pkl", "rb") as f:
    label_classes = pickle.load(f)

# === Camera Initialization ===
picam2 = Picamera2()
picam2.start()
time.sleep(1)

# === Helper Functions ===
def beep():
    pygame.mixer.music.play()
    time.sleep(1)

def speak(text):
    print("🤖", text)
    engine.say(text)
    engine.runAndWait()

def listen():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("🎤 Listening...")
        recognizer.adjust_for_ambient_noise(source)
        try:
            audio = recognizer.listen(source, timeout=5)
            return recognizer.recognize_google(audio).lower()
        except:
            return ""

def move_forward():
    GPIO.output(IN1, GPIO.HIGH)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.HIGH)
    GPIO.output(IN4, GPIO.LOW)

def move_backward():
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.HIGH)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.HIGH)

def stop():
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.LOW)

def check_distance():
    GPIO.output(TRIG, False)
    time.sleep(0.05)
    GPIO.output(TRIG, True)
    time.sleep(0.00001)
    GPIO.output(TRIG, False)

    pulse_start = time.time()
    while GPIO.input(ECHO) == 0:
        pulse_start = time.time()

    pulse_end = time.time()
    while GPIO.input(ECHO) == 1:
        pulse_end = time.time()

    duration = pulse_end - pulse_start
    distance = round(duration * 17150, 2)  # cm
    return distance

def describe_robot(name):
    for robot in robot_data:
        if name.lower() in robot["name"].lower():
            desc = f"{robot['name']} is manufactured by {robot['manufacturer']}. "
            if "weight_kg" in robot:
                desc += f"It weighs {robot['weight_kg']} kilograms. "
            if "payload_kg" in robot:
                desc += f"It can carry a payload of {robot['payload_kg']} kilograms. "
            if "degrees_of_freedom" in robot:
                desc += f"It has {robot['degrees_of_freedom']} degrees of freedom. "
            if "power_source" in robot:
                desc += f"It is powered by {robot['power_source']}. "
            if "applications" in robot:
                desc += f"It is used for {', '.join(robot['applications'])}. "
            if "control_methods" in robot:
                desc += f"It can be controlled using {', '.join(robot['control_methods'])}. "
            if "functions" in robot:
                desc += f"Its key functions include {', '.join(robot['functions'])}. "
            speak(desc)
            return
    speak("Sorry, I don't have information about this robot.")

def predict_robot():
    request = picam2.capture_request()
    frame = request.make_array("main")
    request.release()

    if frame is None or frame.size == 0:
        return None, 0.0

    image = cv2.resize(frame[:, :, :3], (224, 224)) / 255.0
    image = np.expand_dims(image, axis=0)
    predictions = model.predict(image)[0]
    idx = np.argmax(predictions)
    confidence = predictions[idx]
    return label_classes[idx], confidence

# === Main Logic ===
def main():
    beep()
    speak("Hello! I am your Department Guide Robot. I can recognize Kinova, AR100, and Transmission Line robots. Please say the name of the robot or wait for me to detect one.")

    initial_command = listen()
    if any(robot.lower() in initial_command for robot in label_classes):
        describe_robot(initial_command)

    direction = "forward"

    while True:
        dist = check_distance()
        if dist < 20:
            speak("Too close! Reversing.")
            stop()
            move_backward()
            time.sleep(2)
            stop()
            continue

        speak("Detecting robot now.")
        name, confidence = predict_robot()

        if name and confidence > 0.4:
            speak(f"I think this is {name}. Let me tell you more.")
            describe_robot(name)
        else:
            speak("Sorry, I could not recognize the robot.")
            continue

        speak("Say stop, tell me about another robot, or say robot to continue.")
        user_cmd = listen()

        if "stop" in user_cmd:
            speak("Okay, stopping.")
            stop()
            break
        elif "another" in user_cmd or "about" in user_cmd:
            speak("Which robot?")
            robot_name = listen()
            describe_robot(robot_name)
        elif "robot" in user_cmd:
            speak("Okay, moving forward.")
            move_forward()
            time.sleep(2)
            stop()
        else:
            speak("I will now move backward.")
            move_backward()
            time.sleep(2)
            stop()

# === Run ===
try:
    main()
except KeyboardInterrupt:
    stop()
    GPIO.cleanup()
    picam2.stop()
    print("Program stopped.")
